In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedKFold, cross_val_score

#loading data with columnbs
columns = [
    "COMPACTNESS", "CIRCULARITY", "DISTANCE_CIRCULARITY", "RADIUS_RATIO",
    "PR_AXIS_ASPECT_RATIO", "MAX_LENGTH_ASPECT_RATIO", "SCATTER_RATIO",
    "ELONGATEDNESS", "PR_AXIS_RECTANGULARITY", "MAX_LENGTH_RECTANGULARITY",
    "SCALED_VARIANCE_MAJOR", "SCALED_VARIANCE_MINOR", "SCALED_RADIUS_GYRATION",
    "SKEWNESS_MAJOR", "SKEWNESS_MINOR", "KURTOSIS_MINOR", "KURTOSIS_MAJOR",
    "HOLLOWS_RATIO", "target"
]
files = ["xaa.dat", "xab.dat", "xac.dat", 'xad.dat', 'xae.dat', 'xaf.dat', 'xag.dat', 'xah.dat', 'xai.dat']
df = pd.concat([pd.read_csv(f, sep=r"\s+", header=None, names=columns) for f in files],
    ignore_index=True)

# 2) Drop rows containing any missing values
#df = df.dropna()

# 3) Drop irrelevant columns (e.g., ID)
#df = df.drop(columns=['id'], errors='ignore')

# 4) Separate target and features
y = df['target']
X = df.drop(columns=['target'])

# 5) Encode categorical features
for col in X.select_dtypes(include='object').columns:
  X[col] = LabelEncoder().fit_transform(X[col])

# 6) Encode the target if it is a string
if y.dtype == 'object':
  y = LabelEncoder().fit_transform(y)

# 7) Scale numerical features (important for KNN and MLP)
X_scaled = StandardScaler().fit_transform(X)
print('Shape:', X_scaled.shape, 'Classes:', set(y))

Shape: (846, 18) Classes: {'bus', 'saab', 'van', 'opel'}


part2

In [6]:
models = {
    'Linear Classifier' : Perceptron(max_iter=1000, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN' : KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB' : GaussianNB(),
    'Neural Network' : MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42),
}

# 10-fold CV repeated 100 times (1,000 train/eval rounds in total)
rkf = RepeatedKFold(n_splits=10, n_repeats=100, random_state=42)
results = {}
for name, model in models.items():
  scores = cross_val_score(model, X_scaled, y, cv=rkf, scoring='accuracy', n_jobs=-1)
  results[name] = (scores.mean(), scores.std())
  print(f'{name:20s} mean={scores.mean():.4f} std={scores.std():.4f}')

Linear Classifier    mean=0.7293 std=0.0553
Logistic Regression  mean=0.7906 std=0.0433
KNN                  mean=0.7127 std=0.0461
Gaussian NB          mean=0.4575 std=0.0533
Neural Network       mean=0.8447 std=0.0370


part3

In [8]:
from sklearn.feature_selection import SequentialFeatureSelector

models = {
    'Linear Classifier' : Perceptron(max_iter=1000, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN' : KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB' : GaussianNB(),
    'Neural Network' : MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42),
}
feature_names = [
    "COMPACTNESS", "CIRCULARITY", "DISTANCE_CIRCULARITY", "RADIUS_RATIO",
    "PR_AXIS_ASPECT_RATIO", "MAX_LENGTH_ASPECT_RATIO", "SCATTER_RATIO",
    "ELONGATEDNESS", "PR_AXIS_RECTANGULARITY", "MAX_LENGTH_RECTANGULARITY",
    "SCALED_VARIANCE_MAJOR", "SCALED_VARIANCE_MINOR", "SCALED_RADIUS_GYRATION",
    "SKEWNESS_MAJOR", "SKEWNESS_MINOR", "KURTOSIS_MINOR", "KURTOSIS_MAJOR",
    "HOLLOWS_RATIO"
    ]

# Forward selection example (Logistic Regression as the wrapper)
sfs = SequentialFeatureSelector(estimator=models['Linear Classifier'],
                                n_features_to_select='auto', # or an integer
                                direction='forward', # 'backward' is also valid
                                scoring='accuracy',
                                cv=RepeatedKFold(n_splits=10, n_repeats=10, random_state=42),
                                n_jobs=-1,
                                )
sfs.fit(X_scaled, y)
lcselected = np.array(feature_names)[sfs.get_support()]
print('Selected lc features:', list(lcselected))

# Forward selection example (Logistic Regression as the wrapper)
sfs = SequentialFeatureSelector(estimator=LogisticRegression(max_iter=1000),
                                n_features_to_select='auto', # or an integer
                                direction='forward', # 'backward' is also valid
                                scoring='accuracy',
                                cv=RepeatedKFold(n_splits=10, n_repeats=10, random_state=42),
                                n_jobs=-1,
                                )
sfs.fit(X_scaled, y)
lgselected = np.array(feature_names)[sfs.get_support()]
print('Selected lg features:', list(lgselected))

# Forward selection example (Logistic Regression as the wrapper)
sfs = SequentialFeatureSelector(estimator=models['KNN'],
                                n_features_to_select='auto', # or an integer
                                direction='forward', # 'backward' is also valid
                                scoring='accuracy',
                                cv=RepeatedKFold(n_splits=10, n_repeats=10, random_state=42),
                                n_jobs=-1,
                                )
sfs.fit(X_scaled, y)
knselected = np.array(feature_names)[sfs.get_support()]
print('Selected kn features:', list(knselected))

# Forward selection example (Logistic Regression as the wrapper)
sfs = SequentialFeatureSelector(estimator=models['Gaussian NB'],
                                n_features_to_select='auto', # or an integer
                                direction='forward', # 'backward' is also valid
                                scoring='accuracy',
                                cv=RepeatedKFold(n_splits=10, n_repeats=10, random_state=42),
                                n_jobs=-1,
                                )
sfs.fit(X_scaled, y)
gnselected = np.array(feature_names)[sfs.get_support()]
print('Selected gn features:', list(gnselected))

# Forward selection example (Logistic Regression as the wrapper)
print('wait...')
sfs = SequentialFeatureSelector(estimator=models['Neural Network'],
                                n_features_to_select='auto', # or an integer
                                direction='forward', # 'backward' is also valid
                                scoring='accuracy',
                                cv=RepeatedKFold(n_splits=5, n_repeats=3, random_state=42),
                                n_jobs=-1,
                                )
print('heresf')
sfs.fit(X_scaled, y)
print('opps')
nnselected = np.array(feature_names)[sfs.get_support()]
print('Selected nw features:', list(nnselected))



Selected lc features: [np.str_('COMPACTNESS'), np.str_('DISTANCE_CIRCULARITY'), np.str_('SCATTER_RATIO'), np.str_('ELONGATEDNESS'), np.str_('MAX_LENGTH_RECTANGULARITY'), np.str_('SCALED_RADIUS_GYRATION'), np.str_('SKEWNESS_MAJOR'), np.str_('KURTOSIS_MAJOR'), np.str_('HOLLOWS_RATIO')]
Selected lg features: [np.str_('COMPACTNESS'), np.str_('CIRCULARITY'), np.str_('DISTANCE_CIRCULARITY'), np.str_('MAX_LENGTH_ASPECT_RATIO'), np.str_('MAX_LENGTH_RECTANGULARITY'), np.str_('SCALED_VARIANCE_MAJOR'), np.str_('SKEWNESS_MAJOR'), np.str_('SKEWNESS_MINOR'), np.str_('KURTOSIS_MAJOR')]
Selected kn features: [np.str_('COMPACTNESS'), np.str_('CIRCULARITY'), np.str_('RADIUS_RATIO'), np.str_('PR_AXIS_ASPECT_RATIO'), np.str_('MAX_LENGTH_ASPECT_RATIO'), np.str_('SCATTER_RATIO'), np.str_('ELONGATEDNESS'), np.str_('MAX_LENGTH_RECTANGULARITY'), np.str_('SCALED_RADIUS_GYRATION')]
Selected gn features: [np.str_('COMPACTNESS'), np.str_('PR_AXIS_ASPECT_RATIO'), np.str_('ELONGATEDNESS'), np.str_('MAX_LENGTH_RECTAN

it took super long that it kept disconnecting in collab so I just put the result into next cell

In [10]:
lc_sub = [np.str_('COMPACTNESS'), np.str_('DISTANCE_CIRCULARITY'), np.str_('SCATTER_RATIO'), np.str_('ELONGATEDNESS'), np.str_('MAX_LENGTH_RECTANGULARITY'), np.str_('SCALED_RADIUS_GYRATION'), np.str_('SKEWNESS_MAJOR'), np.str_('KURTOSIS_MAJOR'), np.str_('HOLLOWS_RATIO')]
lg_sub = [np.str_('COMPACTNESS'), np.str_('CIRCULARITY'), np.str_('DISTANCE_CIRCULARITY'), np.str_('MAX_LENGTH_ASPECT_RATIO'), np.str_('MAX_LENGTH_RECTANGULARITY'), np.str_('SCALED_VARIANCE_MAJOR'), np.str_('SKEWNESS_MAJOR'), np.str_('SKEWNESS_MINOR'), np.str_('KURTOSIS_MAJOR')]
knn_sub = [np.str_('COMPACTNESS'), np.str_('CIRCULARITY'), np.str_('PR_AXIS_ASPECT_RATIO'), np.str_('MAX_LENGTH_ASPECT_RATIO'), np.str_('SCATTER_RATIO'), np.str_('ELONGATEDNESS'), np.str_('MAX_LENGTH_RECTANGULARITY'), np.str_('SCALED_VARIANCE_MINOR'), np.str_('SCALED_RADIUS_GYRATION')]
gn_sub = [np.str_('COMPACTNESS'), np.str_('PR_AXIS_ASPECT_RATIO'), np.str_('ELONGATEDNESS'), np.str_('MAX_LENGTH_RECTANGULARITY'), np.str_('SCALED_VARIANCE_MAJOR'), np.str_('SKEWNESS_MAJOR'), np.str_('SKEWNESS_MINOR'), np.str_('KURTOSIS_MAJOR'), np.str_('HOLLOWS_RATIO')]
nw_sub = [np.str_('COMPACTNESS'), np.str_('RADIUS_RATIO'), np.str_('PR_AXIS_ASPECT_RATIO'), np.str_('MAX_LENGTH_ASPECT_RATIO'), np.str_('PR_AXIS_RECTANGULARITY'), np.str_('MAX_LENGTH_RECTANGULARITY'), np.str_('SKEWNESS_MAJOR'), np.str_('KURTOSIS_MAJOR'), np.str_('HOLLOWS_RATIO')]

subsets = {
    'Linear Classifier' : lc_sub,
    'Logistic Regression': lg_sub,
    'KNN'                : knn_sub,
    'Gaussian NB'        : gn_sub,
    'Neural Network'     : nw_sub,
}

# 10-fold CV repeated 100 times (1,000 train/eval rounds in total)
rkf = RepeatedKFold(n_splits=10, n_repeats=100, random_state=42)
results_sub = {}
for name, sub in subsets.items():
    # convert feature names → column index positions
    indices = [feature_names.index(col) for col in sub]
    xsub = X_scaled[:, indices]
    scores = cross_val_score(models[name], xsub, y, cv=rkf, scoring='accuracy', n_jobs=-1)
    results_sub[name] = (scores.mean(), scores.std())
    print(f'{name:20s} features={len(sub):2d} mean={scores.mean():.4f} std={scores.std():.4f}')

Linear Classifier    features= 9 mean=0.6703 std=0.0641
Logistic Regression  features= 9 mean=0.7340 std=0.0437
KNN                  features= 9 mean=0.7412 std=0.0471
Gaussian NB          features= 9 mean=0.5587 std=0.0507
Neural Network       features= 9 mean=0.8291 std=0.0383
